<a href="https://colab.research.google.com/github/yashikasgh/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashikasgh/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

My lane is **Content Refresh / Opportunity Scoring**. The goal is to rank pages for editorial review, not to claim that refreshing a page will definitely improve its performance.

I will start with **Logistic Regression because it is relatively interpretable and provides probabilities that can be used to rank pages**. I will compare it with my Week-4 rule-based baseline, which scores pages using freshness, search visibility, and click-through rate (CTR).

I will evaluate both methods on the same held-out data using ranking metrics such as Precision@20 and Precision@50, where appropriate. I will also report the observed positive-label rate and inspect incorrect predictions before interpreting the results.

The target label and feature set will be checked for leakage before training. Model complexity will only be increased if the validation results justify it. These scores are intended to support editorial decisions, not to prove that a content refresh causes traffic growth.

In [4]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Dataset shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
import pandas as pd
import numpy as np

# Check that the dataset is loaded
print("Dataset shape:", df.shape)

# Check missing values in important columns
important_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_90d",
    "days_since_last_update",
    "ctr"
]

print("\nMissing values:")
print(df[important_cols].isna().sum())

# Inspect the observed label distribution
print("\nTrend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nTrend direction proportions:")
print(
    df["trend_direction"]
    .value_counts(normalize=True, dropna=False)
    .round(3)
)

# Inspect baseline signals
print("\nBaseline signal checks:")
print("Unique clients:", df["client_id"].nunique())
print("Unique content IDs:", df["content_id"].nunique())
print("Median impressions:", df["impressions_90d"].median())
print("Median CTR:", df["ctr"].median())

Dataset shape: (30000, 44)

Missing values:
content_id                   0
client_id                    0
trend_direction              0
trend_pct                 3388
impressions_90d              0
days_since_last_update       0
ctr                          0
dtype: int64

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend direction proportions:
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64

Baseline signal checks:
Unique clients: 32
Unique content IDs: 30000
Median impressions: 731.0
Median CTR: 0.07


## 2. Split design

The dataset contains 30,000 content records across 32 clients. Because content performance can share client-specific patterns, a random split may overestimate generalization.

I will use a client-grouped train/test split so that the test set contains clients not seen during training. This tests whether the model generalizes to unseen clients, rather than only to additional pages from familiar clients.

The split will be fixed with a random seed for reproducibility. I will check the target distribution in both partitions and report the positive-label rate alongside the evaluation metrics.

The Week-4 baseline and the learned model must be evaluated on the same test rows. Any preprocessing, imputation, or encoding learned from data will be fitted using training data only.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

# Use the observed trend as a temporary label for split inspection.
# We will verify the label and features before model training.
df_model = df.copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df_model, groups=df_model["client_id"])
)

train_df = df_model.iloc[train_idx].copy()
test_df = df_model.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print("\nClient overlap:")
print(
    set(train_df["client_id"]).intersection(
        set(test_df["client_id"])
    )
)

print("\nTrain trend distribution:")
print(train_df["trend_direction"].value_counts(normalize=True).round(3))

print("\nTest trend distribution:")
print(test_df["trend_direction"].value_counts(normalize=True).round(3))

Train rows: 22885
Test rows: 7115

Train clients: 24
Test clients: 8

Client overlap:
set()

Train trend distribution:
trend_direction
down      0.550
stable    0.194
up        0.138
new       0.084
flat      0.033
Name: proportion, dtype: float64

Test trend distribution:
trend_direction
down      0.517
stable    0.213
up        0.171
flat      0.056
new       0.044
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

I use the Week-4 rule-based score as the baseline and Logistic Regression as the first learned model. For this initial comparison, the observed label is whether `trend_direction` equals `down`. This is a proxy for observed decline, not a label for whether a refresh will succeed.

Both methods are evaluated on the same client-grouped test set. I compare Precision@20 and Precision@50, and report the positive-label rate as context. The baseline ranks pages using freshness, search visibility, and CTR; the model estimates the probability of the observed decline label.

The features are selected conservatively to avoid directly using `trend_pct`, `trend_direction`, or the last-30-day and previous-30-day metrics that may define the trend label. Because the exact construction of `trend_direction` must be confirmed, remaining overlap between the label window and aggregate performance features is a limitation to disclose.

In [7]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# Work on copies so the original dataset stays unchanged
model_df = df.copy()

# Observed label: is the recorded trend "down"?
model_df["is_down"] = (model_df["trend_direction"] == "down").astype(int)

# Recreate the Week-4 median-threshold baseline consistently
model_df["stale"] = (
    model_df["days_since_last_update"] >= 180
).astype(int)

model_df["high_visibility"] = (
    model_df["impressions_90d"] >= model_df["impressions_90d"].median()
).astype(int)

model_df["low_ctr"] = (
    model_df["ctr"] < model_df["ctr"].median()
).astype(int)

model_df["baseline_score"] = (
    40 * model_df["stale"]
    + 40 * model_df["high_visibility"]
    + 20 * model_df["low_ctr"]
)

# Conservative initial feature set.
# Excludes trend label, trend_pct, and last/previous 30-day metrics.
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

feature_cols = numeric_features + categorical_features

# Use the already-created client-grouped split
train_data = model_df.iloc[train_idx].copy()
test_data = model_df.iloc[test_idx].copy()

X_train = train_data[feature_cols]
y_train = train_data["is_down"]

X_test = test_data[feature_cols]
y_test = test_data["is_down"]

# Preprocessing is fitted only on training data
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )),
])

model.fit(X_train, y_train)

# Probability scores for ranking
model_scores = model.predict_proba(X_test)[:, 1]

# Baseline scores on exactly the same test rows
baseline_scores = test_data["baseline_score"].to_numpy()

def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))
    top_idx = np.argsort(scores)[::-1][:k]
    return np.asarray(y_true)[top_idx].mean()

# Same ranking metrics for both methods
results = pd.DataFrame([
    {
        "method": "Week-4 rule baseline",
        "Precision@20": precision_at_k(y_test, baseline_scores, 20),
        "Precision@50": precision_at_k(y_test, baseline_scores, 50),
        "Average precision": average_precision_score(y_test, baseline_scores),
    },
    {
        "method": "Logistic Regression",
        "Precision@20": precision_at_k(y_test, model_scores, 20),
        "Precision@50": precision_at_k(y_test, model_scores, 50),
        "Average precision": average_precision_score(y_test, model_scores),
    },
])

print("Train positive-label rate:", round(y_train.mean(), 3))
print("Test positive-label rate:", round(y_test.mean(), 3))
print("\nModel vs baseline — client-grouped test set")
display(results.round(3))

Train positive-label rate: 0.55
Test positive-label rate: 0.517

Model vs baseline — client-grouped test set


,method,Precision@20,Precision@50,Average precision
0,Week-4 rule baseline,0.6,0.56,0.520
1,Logistic Regression,0.3,0.38,0.514


## 4. Errors and interpretation

On the client-grouped test set, the Week-4 rule-based baseline outperformed Logistic Regression at Precision@20 and Precision@50. Average precision was similar, with the baseline scoring 0.520 and Logistic Regression scoring 0.514.

This suggests that the existing rules were more effective at placing observed declining pages near the top of this particular test ranking. The learned model did not demonstrate an improvement over the baseline on these metrics.

The results are directional: only 8 clients were held out, and the label represents an observed trend rather than the outcome of a content refresh. I will inspect incorrect predictions and model coefficients to understand where the model struggles. Feature associations should not be interpreted as causal explanations.

In [8]:
# Threshold-based errors for inspecting the binary "down" label
test_results = test_data[
    ["content_id", "client_id", "trend_direction", "trend_pct",
     "impressions_90d", "days_since_last_update", "ctr",
     "baseline_score"]
].copy()

test_results["actual_down"] = y_test.to_numpy()
test_results["model_probability"] = model_scores
test_results["predicted_down"] = (model_scores >= 0.5).astype(int)

# False positives: predicted down, but observed label was not down
false_positives = test_results[
    (test_results["predicted_down"] == 1) &
    (test_results["actual_down"] == 0)
]

# False negatives: observed down, but model did not predict down
false_negatives = test_results[
    (test_results["predicted_down"] == 0) &
    (test_results["actual_down"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(false_positives.head(3))

print("\nExample false negatives:")
display(false_negatives.head(3))

# Inspect model coefficients after one-hot encoding
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

coef_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

print("\nTop features associated with predicting 'down':")
display(
    coef_table.sort_values("coefficient", ascending=False).head(10)
)

print("\nTop features associated with predicting 'not down':")
display(
    coef_table.sort_values("coefficient", ascending=True).head(10)
)

False positives: 1512
False negatives: 1973

Example false positives:


,content_id,client_id,trend_direction,trend_pct,impressions_90d,days_since_last_update,ctr,baseline_score,actual_down,model_probability,predicted_down
13,content_a5a2fbc76336,client_8527a891e2,stable,10.4,307,103,0.00,20,0,0.689138,1
56,content_dcebfd222b10,client_f369cb89fc,up,50.0,16,20,0.00,20,0,0.590534,1
64,content_685de0e3b7cb,client_f369cb89fc,up,45.2,2639,8,0.11,40,0,0.578773,1



Example false negatives:


,content_id,client_id,trend_direction,trend_pct,impressions_90d,days_since_last_update,ctr,baseline_score,actual_down,model_probability,predicted_down
0,content_304f48230142,client_f369cb89fc,down,-41.4,3803,20,0.76,40,1,0.454787,0
1,content_a1fb4e703a9e,client_4e07408562,down,-57.7,15320,25,0.05,60,1,0.389434,0
19,content_af865035b328,client_f369cb89fc,down,-67.7,99,20,2.02,0,1,0.440976,0



Top features associated with predicting 'down':


,feature,coefficient
20,cat__freshness_tier_0-30,0.508972
22,cat__freshness_tier_31-90,0.451667
6,num__days_since_last_update,0.391781
11,cat__content_type_keyword article,0.374309
23,cat__freshness_tier_91-180,0.283986
3,num__word_count,0.236475
13,cat__main_intent_informational,0.230173
17,cat__age_tier_31-90,0.155053
24,cat__word_count_tier_1000-2000,0.143421
30,cat__char_count_tier_8000-15000,0.101005



Top features associated with predicting 'not down':


,feature,coefficient
21,cat__freshness_tier_181+,-1.539585
10,cat__content_type_feedly article,-0.669269
14,cat__main_intent_navigational,-0.655122
5,num__content_age_days,-0.421282
16,cat__age_tier_181-365,-0.338899
27,cat__word_count_tier_<1000,-0.264388
4,num__char_count,-0.231472
8,cat__competition_level_LOW,-0.200997
31,cat__char_count_tier_<8000,-0.182606
28,cat__char_count_tier_15000-25000,-0.143727


The error inspection found both false positives and false negatives at the 0.5 probability threshold. For example, one page with an observed `stable` trend received a predicted probability of approximately 0.689, while an observed `down` page received approximately 0.455 and was classified as not down.

The largest positive Logistic Regression coefficients included freshness categories, days since last update, and content-related features. These are model associations, not causal explanations. The positive association for the `freshness_tier_0-30` category is unexpected and warrants checking the category definitions and label construction.

Overall, Logistic Regression did not outperform the Week-4 baseline on the reported ranking metrics. The baseline remains the stronger method in this evaluation, and the model's errors show that its scores should be used cautiously as decision support rather than as definitive refresh recommendations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.